# CIFAR-10 Image Classification with Transfer Learning (ResNet50)

## Project goal
We build an end-to-end image classification pipeline for **CIFAR-10** (10 classes, 32×32 RGB images) using **transfer learning** with a pretrained **ResNet50** model.

**What matters for this project**
- A correct, reproducible workflow (data → preprocessing → model → training → evaluation).
- Clear reasoning and documented decisions (why we do each step).
- Observations during training (stability, overfitting, trends), not only the final accuracy.

**Compute note**
Training a CNN (especially fine-tuning) can take a long time on CPU. In Colab, switching runtime to GPU can reduce training time significantly.


## 0. Plan & deliverables

**Deliverables in this notebook**
1. Dataset loading + sampling logic (`n=10000` train)
2. EDA and sanity checks (shapes, labels, class balance, example images)
3. Preprocessing pipeline compatible with ResNet50
4. Model architecture (ResNet50 base + custom head)
5. Two-stage training (freeze → train head, then unfreeze → fine-tune)
6. Evaluation on the test set + interpretation
7. A short conclusion section to translate into Google Slides

**Observations log (we will update as we go)**
- Decisions made and why
- Training behavior (learning curves)
- Limitations (runtime, hardware)
- What we would improve next


## 1. Setup: imports, reproducibility, and hardware

**Why this section exists**
- Imports: everything needed in one place.
- Reproducibility: seeding reduces randomness (not perfect with GPU, but still useful).
- Hardware check: training speed depends heavily on GPU availability.


### Colab runtime checklist (GPU + TensorFlow)

**Goal**: make sure we run on GPU and TensorFlow is installed before importing it.

1. `Runtime` → `Change runtime type` → `Hardware accelerator` → **GPU**
2. If TensorFlow import fails, the next cell will install it automatically.

After running the next cell, you should see:
- TensorFlow version printed
- `Num GPUs Available` is `> 0` (if GPU is enabled)


### What we install/import (and why)

- **NumPy**: numerical arrays; used for label counts and basic manipulation.
- **Matplotlib**: visualization of sample images and training curves.
- **TensorFlow / Keras**: deep learning framework; provides CIFAR-10 loader, ResNet50, training API.

We install TensorFlow only if it is missing in the current runtime.


### TensorFlow installation (local Jupyter)

If you are running this notebook **locally in Jupyter** (not Google Colab), TensorFlow must be installed into the **same Python environment / kernel** that Jupyter uses.

**Why installation sometimes 'does not work'**
- You installed TensorFlow into a different Python (e.g., system Python), while Jupyter runs from a virtualenv/conda env.
- No internet access / blocked network.
- Platform-specific build issues (macOS Intel vs Apple Silicon).

**Step 1: confirm which Python your notebook uses** (run the next cell).
**Step 2: install TensorFlow using that exact Python**.


In [ ]:
import sys
print('Python executable used by THIS notebook:')
print(sys.executable)

# After you see the path above, install TensorFlow in a terminal using the SAME python:
#   <that_python> -m pip install -U pip
#   <that_python> -m pip install tensorflow
# Example:
#   /path/to/python -m pip install tensorflow


In [ ]:
# Standard library
import os          # OS utilities (paths, environment variables).
import random      # Randomness control for reproducibility (Python-level RNG).

# Numeric computing
import numpy as np # Arrays + fast numeric ops; used for label distribution and basic data handling.

# Visualization
import matplotlib.pyplot as plt  # Plot sample images and training curves.

# Deep Learning framework
import tensorflow as tf  # TensorFlow provides Keras API, pretrained ResNet50, training loops, GPU support.

# Local (no-network) CIFAR-10 loader
import sys
sys.path.insert(0, 'scripts')
from local_cifar10 import load_cifar10_from_tar

# Keras utilities we will use
from tensorflow.keras import layers, models                        # Layers + Model API.
from tensorflow.keras.applications import ResNet50                 # Pretrained backbone.
from tensorflow.keras.applications.resnet50 import preprocess_input # Correct preprocessing for ResNet50 ImageNet weights.

# Make Matplotlib cache writable in this repo (avoids slow / permission warnings)
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('artifacts/mplconfig'))

print('TensorFlow:', tf.__version__)
print('Num GPUs Available:', len(tf.config.list_physical_devices('GPU')))


## 2. Load CIFAR-10 and limit training set to 10k

**Why we limit the train set**
- CIFAR-10 has 50k training images. For this sprint, we use **10,000** to keep training time manageable.
- We limit *once* at the start to keep the rest of the notebook consistent.

**Note about splits**
We will create our own **train/validation/test** split from this 10k subset (stratified). We do **not** use the official CIFAR-10 test set in this notebook to avoid mixing evaluation protocols.

**What we expect (before splitting)**
- `train_images_full`: `(10000, 32, 32, 3)`
- `train_labels_full`: `(10000, 1)`


In [ ]:
data = load_cifar10_from_tar('.keras/datasets/cifar-10-python.tar.gz')
(train_images_full, train_labels_full), _ = (data.x_train, data.y_train), (data.x_test, data.y_test)

n = 10_000
train_images_full = train_images_full[:n]
train_labels_full = train_labels_full[:n]

print('train_images_full:', train_images_full.shape, train_images_full.dtype)
print('train_labels_full:', train_labels_full.shape, train_labels_full.dtype)


## 2.5. Stratified split: train / validation / test

We take the **10,000**-image subset and split it into:
- **90% train**
- **3% validation**
- **7% test**

We **stratify by label** so each split keeps (approximately) the same class proportions.


In [ ]:
from sklearn.model_selection import train_test_split

# We split ONLY the 10k subset (no leakage from the official CIFAR-10 test set)
X = train_images_full
y = train_labels_full

y_flat = y.reshape(-1)

# 1) Hold out 7% test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.07,
    random_state=42,
    stratify=y_flat
)

# 2) From the remaining 93%, take 3% validation (=> 3/93 of trainval)
val_frac_of_trainval = 0.03 / (1.0 - 0.07)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=val_frac_of_trainval,
    random_state=42,
    stratify=y_trainval.reshape(-1)
)

print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_val  :', X_val.shape, 'y_val  :', y_val.shape)
print('X_test :', X_test.shape, 'y_test :', y_test.shape)

# Sanity-check class balance
for name, yy in [('train', y_train), ('val', y_val), ('test', y_test)]:
    unique, counts = np.unique(yy.reshape(-1), return_counts=True)
    dist = dict(zip(unique.tolist(), counts.tolist()))
    print(name, dist)


## 3. EDA: labels, class names, and sample images

**Why EDA matters (even for deep learning)**
- Ensures labels are in the expected format.
- Checks class distribution (sampling bugs happen).
- Confirms images look reasonable (RGB, no corruption).

We will:
- define class names,
- check label distribution,
- visualize a small grid of images.


In [ ]:
train_labels_flat = y_train.flatten()

unique, counts = np.unique(train_labels_flat, return_counts=True)
for u, c in zip(unique, counts):
    print(f'{u}: {class_names[u]:>10} -> {c}')


In [ ]:
show_image_grid(X_train, y_train.flatten(), class_names)


## 4. Preprocessing for ResNet50

**Key decision**: use `tf.keras.applications.resnet50.preprocess_input`.

**Why**
ResNet50 pretrained weights expect inputs preprocessed in a specific way. Using the matching preprocessing:
- aligns our data distribution with what the base model expects,
- improves transfer learning stability.

We will keep two versions:
- raw images for visualization (`uint8`),
- preprocessed images for the model (`float32`).


In [ ]:
X_train_raw = X_train
X_val_raw = X_val
X_test_raw = X_test

X_train_pp = preprocess_input(X_train_raw.astype('float32'))
X_val_pp = preprocess_input(X_val_raw.astype('float32'))
X_test_pp = preprocess_input(X_test_raw.astype('float32'))

print('X_train_pp:', X_train_pp.shape, X_train_pp.dtype)
print('pp value range (train):', float(X_train_pp.min()), 'to', float(X_train_pp.max()))


## 5. Model: ResNet50 base + custom head

**Transfer learning structure**
- **Base model**: ResNet50 pretrained on ImageNet (`weights='imagenet'`).
- **Head**: trainable classifier for 10 CIFAR-10 classes.

**Why freeze first**
Freezing the base:
- makes training faster,
- reduces the risk of destroying pretrained features early,
- lets the head adapt to the new task first.


In [ ]:
base_model = ResNet50(
    weights='.keras/models/resnet50_notop.h5',
    include_top=False,
    input_shape=(32, 32, 3)
)
base_model.trainable = False

inputs = layers.Input(shape=(32, 32, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.summary()


## 6. Compile

**Loss & labels**
CIFAR-10 labels are integers, so we use `SparseCategoricalCrossentropy`.

**Metrics**
We track accuracy to measure classification performance.

We start with Adam and a moderate learning rate for head training.


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)


## 7. Stage 1 training: train the head (base frozen)

**Goal**
Learn a classifier head on top of fixed pretrained features.

**Runtime note**
This should be faster than fine-tuning.


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
]

history_head = model.fit(
    X_train_pp, y_train,
    validation_data=(X_val_pp, y_val),
    epochs=10,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)


### Training curves

We plot accuracy/loss curves to understand training dynamics:
- Is validation improving?
- Do we see overfitting (train ↑ while val ↓)?
- Does fine-tuning provide an additional gain?


In [ ]:
def plot_history(history, title_prefix=''):
    hist = history.history
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(hist.get('accuracy', []), label='train')
    plt.plot(hist.get('val_accuracy', []), label='val')
    plt.title(f'{title_prefix} Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(hist.get('loss', []), label='train')
    plt.plot(hist.get('val_loss', []), label='val')
    plt.title(f'{title_prefix} Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

plot_history(history_head, title_prefix='Stage 1 (Head-only)')


## 8. Stage 2 training: fine-tuning (unfreeze base)

**Goal**
Adapt the pretrained backbone to CIFAR-10.

**Key rule**
Use a smaller learning rate during fine-tuning.


In [ ]:
# Unfreeze only the last part of the backbone (faster on CPU than full fine-tuning)
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

history_ft = model.fit(
    X_train_pp, y_train,
    validation_data=(X_val_pp, y_val),
    epochs=10,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
plot_history(history_ft, title_prefix='Stage 2 (Fine-tuning)')


## 9. Evaluation: metrics on train / validation / test

We report metrics on all three splits to see generalization clearly.


In [ ]:
def eval_split(name, X_pp, y_true):
    loss, acc = model.evaluate(X_pp, y_true, verbose=0)
    print(f'{name:>5} -> loss: {loss:.4f} | acc: {acc:.4f}')

eval_split('train', X_train_pp, y_train)
eval_split('val', X_val_pp, y_val)
eval_split('test', X_test_pp, y_test)


### Observation notes (fill in after training)

- Stage 1: (e.g., fast gains early, then plateau)
- Stage 2: (e.g., slower but continued improvement / instability)
- Overfitting signs: (if any)
- Runtime notes: (CPU/GPU, total time)


## 9. Evaluation on the test set

We evaluate on the untouched test set to estimate generalization.


In [ ]:
# Metrics on the held-out test split
test_loss, test_acc = model.evaluate(X_test_pp, y_test, verbose=0)
print('Test loss:', test_loss)
print('Test accuracy:', test_acc)


### Optional: confusion matrix (error analysis)

A confusion matrix can reveal which classes are commonly confused (e.g., cat vs dog, truck vs automobile).
This is helpful for interpretation in the final presentation.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = y_test.flatten()
y_prob = model.predict(X_test_pp, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

plt.figure(figsize=(8, 8))
disp.plot(cmap='Blues', xticks_rotation=45, values_format='d')
plt.title('Confusion Matrix (Test Split)')
plt.show()


## 11. Save artifacts (optional)

Saving artifacts helps reproducibility and makes it easier to reuse results for the presentation.
In Colab, you can also save to Google Drive.


In [ ]:
# Example: save model (uncomment if needed)
# model.save('cifar10_resnet50_transfer_learning.keras')

# Example: save histories for later plotting
# import json
# with open('history_head.json', 'w') as f:
#     json.dump(history_head.history, f)
# with open('history_ft.json', 'w') as f:
#     json.dump(history_ft.history, f)


## 10. Conclusions and next improvements

Write 5–10 bullet points answering:
- What worked well?
- What was hard or slow?
- Did we see overfitting?
- What would we do next with more time/compute?

This section will be reused for the final Google Slides presentation.


## 10. Results summary (English)

This run was executed locally on CPU (no GPU detected by TensorFlow).

We compared two approaches:
1) **Stage 1 (Head-only)**: freeze ResNet50 backbone, train only the custom classifier head.
2) **Stage 2 (Fine-tune last 20 layers)**: unfreeze only the last 20 backbone layers and train with a smaller learning rate.

Below we print the metrics saved in `artifacts/run_summary.json`.


In [ ]:
import json

with open('artifacts/run_summary.json', 'r') as f:
    s = json.load(f)

print('TensorFlow:', s['tensorflow'])
print('GPUs:', s['gpus'])
print('
Stage 1 (head-only):')
print('  epochs_ran   :', s['stage1']['epochs_ran'])
print('  best val_acc :', s['stage1']['val_acc_best'])
print('  best val_loss:', s['stage1']['val_loss_best'])

print('
Stage 2 (fine-tune last 20):')
print('  epochs_ran   :', s['stage2']['epochs_ran'])
print('  best val_acc :', s['stage2']['val_acc_best'])
print('  best val_loss:', s['stage2']['val_loss_best'])

print('
Test set:')
print('  test_accuracy:', s['test_accuracy'])
print('  test_loss    :', s['test_loss'])

print('
Conclusion (short):')
if s['stage2']['val_acc_best'] > s['stage1']['val_acc_best']:
    print('- Fine-tuning improved best validation accuracy.')
else:
    print('- Fine-tuning did not improve best validation accuracy (on this run).')
